# Week 6 Homework: Predicting Wages with Trees and Random Forest

## Purpose of Homework

This homework will give you practice applying **cost-complexity pruning** and **random forests** to a regression problem. You will predict log wages for employed recent graduates using the same CPS data from lecture.

You are encouraged to refer to lecture content and liberally use course resources such as the discussion board and office hours.

## Logistics

Due date: The homework is due **11:59pm on Thursday, March 5, 2026**.

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/).

1. Download this file (`STA272_hw6_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw6 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

All homeworks will take place in a Jupyter notebook (like this one). When you are done, you will download this notebook and submit it to MarkUs.

## About the Data

We use individual-level microdata from the [IPUMS CPS](https://cps.ipums.org/cps/) Annual Social and Economic Supplement (ASEC), 2020–2025. The data has been filtered to **recent graduates**: ages 22–30 with a bachelor's degree or higher.

**Research Question:** What demographic, occupational, and socioeconomic characteristics predict wage income among employed recent graduates?

### Target Variable

| Variable | Description | Type |
|----------|-------------|------|
| `INCWAGE` | Wage and salary income ($) | Continuous |

Because wages are right-skewed, we predict **log(INCWAGE)** and restrict to employed workers with positive wages.

### Predictor Variables

| Variable | Description | Type |
|----------|-------------|------|
| `AGE` | Age in years | Numeric |
| `female` | Sex | Binary (0=Male, 1=Female) |
| `married` | Marital status | Binary (0=Not married, 1=Married) |
| `educ_cat` | Education level | Ordinal (4=Bachelor's, 5=Graduate) |
| `faminc_mid` | Family income (bracket midpoint, $) | Numeric ($2,500–$200,000; $150k+ top-coded at $200k) |
| `metro_binary` | Metropolitan area | Binary |
| `race_cat` | Race | Categorical (White, Black, Asian, Other) |
| `us_citizen` | US citizenship | Binary |
| `has_children` | Has children in household | Binary |
| `insured` | Health insurance | Binary |
| `OCC` | Occupation code | Numeric |
| `IND` | Industry code | Numeric |
| `FIRMSIZE` | Employer firm size | Ordinal (0–9) with 9 being 1000+ employees |
| `NCHILD` | Number of own children | Numeric |
| `YEAR` | Survey year | Numeric |

`OCC` and `IND` codes available [here](https://www.census.gov/programs-surveys/cps/technical-documentation/methodology/industry-and-occupation-classification.html)

## Task #1: Load and Prepare the Data

Load `cps_empl_new.csv` into a pandas dataframe called `cps_df`. Then:
1. Filter to **employed workers with positive wages**: keep rows where `employed == 1` and `INCWAGE > 0`. Store the result in `cps_empl`.
2. Create `log_wage = np.log(cps_empl['INCWAGE'])`.
3. Print the shape of `cps_empl` and the mean wage (in dollars) and mean log-wage.
4. Plot side-by-side histograms of raw `INCWAGE` and `log_wage` (as done in lecture).

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #1 in this cell

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
cps_df = pd.read_csv(...)

# Filter to employed workers with positive wages
cps_empl = cps_df[...].copy()

# Create log_wage
cps_empl['log_wage'] = ...

print(f'Shape of cps_empl: {cps_empl.shape}')
print(f'Mean wage:     ${cps_empl["INCWAGE"].mean():,.0f}')
print(f'Mean log-wage: {cps_empl["log_wage"].mean():.3f}')

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)

axes[0].hist(cps_empl['INCWAGE'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('INCWAGE ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Wages (Raw)')

axes[1].hist(cps_empl['log_wage'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(INCWAGE)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Wages (Log Scale)')

plt.tight_layout()
plt.show()

In [ ]:
# Answer cell

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
cps_df = pd.read_csv('cps_empl_new.csv')

# Filter to employed workers with positive wages
cps_empl = cps_df[(cps_df['employed'] == 1) & (cps_df['INCWAGE'] > 0)].copy()

# Create log_wage
cps_empl['log_wage'] = np.log(cps_empl['INCWAGE'])

print(f'Shape of cps_empl: {cps_empl.shape}')
print(f'Mean wage:     ${cps_empl["INCWAGE"].mean():,.0f}')
print(f'Mean log-wage: {cps_empl["log_wage"].mean():.3f}')

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)

axes[0].hist(cps_empl['INCWAGE'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('INCWAGE ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Wages (Raw)')

axes[1].hist(cps_empl['log_wage'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(INCWAGE)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Wages (Log Scale)')

plt.tight_layout()
plt.show()

## Task #2: Encode Categorical Variable and Set Up Features

Since `race_cat` is a string variable, we need to encode it as dummy variables.

1. Apply `pd.get_dummies()` to `cps_empl` with `columns=['race_cat']` and `drop_first=False`. Store the result in `cps_encoded`.
2. Define `reg_feature_cols` as the list of predictor variable names:
   `['AGE', 'female', 'married', 'educ_cat', 'faminc_mid', 'metro_binary', 'us_citizen', 'has_children', 'insured', 'OCC', 'IND', 'FIRMSIZE', 'NCHILD', 'YEAR', 'race_cat_Asian', 'race_cat_Black', 'race_cat_Other', 'race_cat_White']`
3. Create `X = cps_encoded[reg_feature_cols]` and `y = cps_encoded['log_wage']`.
4. Print the shape of `X`.

**Note on `drop_first`:** In linear regression, one dummy category must be dropped to avoid perfect multicollinearity. For tree-based methods (decision trees and random forests), this is not necessary — trees split on one feature at a time and are unaffected by linear dependencies among predictors. Including all four race dummies allows the tree to split directly on any race group (including Asian) in a single step, rather than having to infer it from the other three dummies all being zero.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #2 in this cell

cps_encoded = pd.get_dummies(cps_empl, columns=..., drop_first=...)

reg_feature_cols = ['AGE', 'female', 'married', 'educ_cat', 'faminc_cat',
                    'metro_binary', 'us_citizen', 'has_children', 'insured',
                    'OCC', 'IND', 'FIRMSIZE', 'NCHILD', 'YEAR',
                    'race_cat_Black', 'race_cat_Other', 'race_cat_White']

X = ...
y = ...

print(f'X shape: {X.shape}')

In [ ]:
# Answer cell

cps_encoded = pd.get_dummies(cps_empl, columns=['race_cat'], drop_first=False)

reg_feature_cols = ['AGE', 'female', 'married', 'educ_cat', 'faminc_mid',
                    'metro_binary', 'us_citizen', 'has_children', 'insured',
                    'OCC', 'IND', 'FIRMSIZE', 'NCHILD', 'YEAR',
                    'race_cat_Asian', 'race_cat_Black', 'race_cat_Other', 'race_cat_White']

X = cps_encoded[reg_feature_cols]
y = cps_encoded['log_wage']

print(f'X shape: {X.shape}')

## Task #3: Train/Test Split

Split the data into training and test sets using `train_test_split` with:
- Test size: 20%
- `random_state=272`

Store the results as `X_train`, `X_test`, `y_train`, `y_test`. Print the sizes of each set.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #3 in this cell

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(..., ..., test_size=..., random_state=...)

print(f'Training set: {len(X_train):,} observations')
print(f'Test set:     {len(X_test):,} observations')
print(f'Mean log-wage (train): {y_train.mean():.3f}  (${np.exp(y_train.mean()):,.0f})')

In [ ]:
# Answer cell

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=272)

print(f'Training set: {len(X_train):,} observations')
print(f'Test set:     {len(X_test):,} observations')
print(f'Mean log-wage (train): {y_train.mean():.3f}  (${np.exp(y_train.mean()):,.0f})')

## Task #4: Fit a Full (Unpruned) Regression Tree

Fit a `DecisionTreeRegressor` with **no constraints** (let the tree grow fully) on the training data. Store it as `dt_full`.

Compute and print:
- The full tree depth and number of leaves
- Training RMSE and R²
- Test RMSE and R²

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #4 in this cell

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Fit full unpruned tree
dt_full = DecisionTreeRegressor(random_state=272)
dt_full.fit(..., ...)


# Training and test metrics
rmse_train_full = np.sqrt(mean_squared_error(y_train, dt_full.predict(X_train)))
rmse_test_full  = np.sqrt(mean_squared_error(y_test, ...))
r2_train_full   = r2_score(y_train, dt_full.predict(X_train))
r2_test_full    = r2_score(y_test, ...)

print(f'Full tree training RMSE:    {rmse_train_full:.3f}  |  R² = {r2_train_full:.3f}')
print(f'Full tree test RMSE:        {rmse_test_full:.3f}  |  R² = {r2_test_full:.3f}')
print(f'Tree depth: {dt_full.get_depth()},  Leaves: {dt_full.get_n_leaves()}')

In [ ]:
# Answer cell

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

dt_full = DecisionTreeRegressor(random_state=272)
dt_full.fit(X_train, y_train)

rmse_train_full = np.sqrt(mean_squared_error(y_train, dt_full.predict(X_train)))
rmse_test_full  = np.sqrt(mean_squared_error(y_test,  dt_full.predict(X_test)))
r2_train_full   = r2_score(y_train, dt_full.predict(X_train))
r2_test_full    = r2_score(y_test,  dt_full.predict(X_test))

print(f'Full tree training RMSE:    {rmse_train_full:.3f}  |  R² = {r2_train_full:.3f}')
print(f'Full tree test RMSE:        {rmse_test_full:.3f}  |  R² = {r2_test_full:.3f}')
print(f'Tree depth: {dt_full.get_depth()},  Leaves: {dt_full.get_n_leaves()}')

## Task #5: Cost-Complexity Pruning with 10-Fold Cross-Validation

Cost-complexity pruning controls tree size via a penalty parameter `ccp_alpha` ($\alpha \geq 0$). Larger $\alpha$ removes more branches, producing a simpler tree.

**Step 1**: Obtain the pruning path using `dt_full.cost_complexity_pruning_path(X_train, y_train)`. This returns an object with `.ccp_alphas` (candidate alpha values) and `.impurities`.

**Step 2**: For each candidate `ccp_alpha`, compute the **10-fold cross-validation RMSE** on the training set using `cross_val_score` with `scoring='neg_mean_squared_error'` and `cv=10`.

**Step 3**: Plot mean CV RMSE vs `ccp_alpha` with error bars (±1 standard deviation). Choose `best_alpha` as the value with the **lowest mean CV RMSE**.

**Note**: The full pruning path can contain thousands of candidate alpha values, making cross-validation very slow. Instead, search over a grid of **25 evenly spaced values** between 0 and the maximum alpha using `np.linspace(0, alpha_max, 25)`. Skip `ccp_alpha = 0` (the unpruned tree) since it may take a long time.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #5 in this cell

from sklearn.model_selection import cross_val_score

# Step 1: Get pruning path
pruning_path = dt_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = pruning_path.ccp_alphas[1:]  # skip alpha=0

# Step 2: 10-fold CV RMSE for each alpha
cv_rmse_mean = []
cv_rmse_std  = []

for alpha in ccp_alphas:
    dt_cv = DecisionTreeRegressor(ccp_alpha=..., random_state=272)
    scores = cross_val_score(dt_cv, X_train, y_train,
                             cv=..., scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-scores)
    cv_rmse_mean.append(rmse_scores.mean())
    cv_rmse_std.append(rmse_scores.std())

cv_rmse_mean = np.array(cv_rmse_mean)
cv_rmse_std  = np.array(cv_rmse_std)

# Step 3: Plot CV RMSE vs ccp_alpha
plt.figure(figsize=(9, 4), dpi=150)
plt.plot(ccp_alphas, cv_rmse_mean, 'o-', markersize=4, label='Mean CV RMSE')
plt.fill_between(ccp_alphas,
                 cv_rmse_mean - cv_rmse_std,
                 cv_rmse_mean + cv_rmse_std,
                 alpha=0.2, label='±1 SD')
plt.xlabel('ccp_alpha')
plt.ylabel('CV RMSE (log-wage scale)')
plt.title('10-Fold CV RMSE vs. Cost-Complexity Pruning Parameter')
plt.legend()
plt.tight_layout()
plt.show()

# Select best alpha
best_alpha = ccp_alphas[np.argmin(cv_rmse_mean)]
print(f'Best ccp_alpha: {best_alpha:.5f}')
print(f'Best CV RMSE:   {cv_rmse_mean.min():.3f}')

In [ ]:
# Answer cell

from sklearn.model_selection import cross_val_score

pruning_path = dt_full.cost_complexity_pruning_path(X_train, y_train)
alpha_max = pruning_path.ccp_alphas.max()
ccp_alphas = np.linspace(0, alpha_max, 25)[1:]  # 24 values, skip alpha=0
print(f"Searching over {len(ccp_alphas)} alpha values (max = {alpha_max:.5f})")

cv_rmse_mean = []
cv_rmse_std  = []

for alpha in ccp_alphas:
    dt_cv = DecisionTreeRegressor(ccp_alpha=alpha, random_state=272)
    scores = cross_val_score(dt_cv, X_train, y_train,
                             cv=10, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-scores)
    cv_rmse_mean.append(rmse_scores.mean())
    cv_rmse_std.append(rmse_scores.std())

cv_rmse_mean = np.array(cv_rmse_mean)
cv_rmse_std  = np.array(cv_rmse_std)

plt.figure(figsize=(9, 4), dpi=150)
plt.plot(ccp_alphas, cv_rmse_mean, 'o-', markersize=4, label='Mean CV RMSE')
plt.fill_between(ccp_alphas,
                 cv_rmse_mean - cv_rmse_std,
                 cv_rmse_mean + cv_rmse_std,
                 alpha=0.2, label='±1 SD')
plt.xlabel('ccp_alpha')
plt.ylabel('CV RMSE (log-wage scale)')
plt.title('10-Fold CV RMSE vs. Cost-Complexity Pruning Parameter')
plt.legend()
plt.tight_layout()
plt.show()

best_alpha = ccp_alphas[np.argmin(cv_rmse_mean)]
print(f'Best ccp_alpha: {best_alpha:.5f}')
print(f'Best CV RMSE:   {cv_rmse_mean.min():.3f}')

## Task #6: Fit and Evaluate the Pruned Tree

Using the `best_alpha` from Task #5:
1. Fit a `DecisionTreeRegressor` with `ccp_alpha=best_alpha` and `random_state=272` on the full training set. Store it as `dt_pruned`.
2. Print the depth and number of leaves of the pruned tree.
3. Compute and print the test RMSE and R² of the pruned tree.
4. Plot the pruned tree using `plot_tree`.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #6 in this cell

from sklearn.tree import plot_tree

# Fit pruned tree
dt_pruned = DecisionTreeRegressor(ccp_alpha=..., random_state=272)
dt_pruned.fit(..., ...)

rmse_pruned = np.sqrt(mean_squared_error(y_test, dt_pruned.predict(...)))
r2_pruned   = r2_score(y_test, dt_pruned.predict(...))

print(f'Pruned tree depth:  {dt_pruned.get_depth()}')
print(f'Pruned tree leaves: {dt_pruned.get_n_leaves()}')
print(f'Pruned tree test RMSE: {rmse_pruned:.3f}  |  R² = {r2_pruned:.3f}')

plt.figure(figsize=(20, 8), dpi=150)
plot_tree(dt_pruned,
          feature_names=...,
          filled=True,
          rounded=True,
          fontsize=8,
          precision=2)
plt.title(f'Pruned Regression Tree (ccp_alpha={best_alpha:.4f}) — Test RMSE: {rmse_pruned:.3f}  R²={r2_pruned:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# Answer cell

from sklearn.tree import plot_tree

dt_pruned = DecisionTreeRegressor(ccp_alpha=best_alpha, random_state=272)
dt_pruned.fit(X_train, y_train)

rmse_pruned = np.sqrt(mean_squared_error(y_test, dt_pruned.predict(X_test)))
r2_pruned   = r2_score(y_test, dt_pruned.predict(X_test))

print(f'Pruned tree depth:  {dt_pruned.get_depth()}')
print(f'Pruned tree leaves: {dt_pruned.get_n_leaves()}')
print(f'Pruned tree test RMSE: {rmse_pruned:.3f}  |  R² = {r2_pruned:.3f}')

plt.figure(figsize=(20, 8), dpi=150)
plot_tree(dt_pruned,
          feature_names=reg_feature_cols,
          filled=True,
          rounded=True,
          fontsize=8,
          precision=2)
plt.title(f'Pruned Regression Tree (ccp_alpha={best_alpha:.4f}) — Test RMSE: {rmse_pruned:.3f}  R²={r2_pruned:.3f}')
plt.tight_layout()
plt.show()

---

## Question #1: Interpreting Cost-Complexity Pruning (4 points)

a) Compare the full tree (Task #4) and the pruned tree (Task #6) in terms of depth, number of leaves, and test RMSE and R2. What does this tell you about the trade-off between tree complexity and generalization? (2 points)

b) Why is it important to select `ccp_alpha` using cross-validation on the **training set** rather than by evaluating RMSE directly on the test set? (1 points)

c) Look at the root split in the pruned tree. Which feature does it split on first? What does this suggest about the most important predictor of wages? (1 point)

### Model Answer:

*a) The full tree grows very deep (Tree depth: 35,  Leaves: 15,148) and achieves very small (0.051) training RMSE but high test RMSE 1.050 and very poor test R² = -0.501, a clear sign of overfitting. The pruned tree has far fewer leaves (13) and a smaller test RMSE 0.772 and better R2=0.188. This illustrates the bias–variance trade-off: a simpler tree has higher bias but lower variance, which often generalizes better to unseen data.*

**Grading note:** Students should note the full tree's very low training RMSE and better R2 vs. higher test RMSE and very poor R2 (overfitting), and that pruning reduces variance. The direction of the comparison must be correct.

*b) If we selected `ccp_alpha` by evaluating on the test set, we would be using the test set to make a modelling decision — effectively making it part of the training process. The test set would no longer provide a fair, independent estimate of generalization error. This is sometimes called "data leakage" from the test set. Cross-validation uses only the training data to estimate generalization error, keeping the test set truly held out.*

**Grading note:** Students must explain that using the test set to tune hyperparameters gives an optimistic/biased estimate of test error. The concept of data leakage or contamination of the test set should be present.

*c) The root split is on age. This suggests that age is the strongest single predictor of log wages — which aligns with intuition about experience. *

**Grading note:** Students should correctly identify the root split variable from their tree plot. Interpretation should connect the variable to wages conceptually.

Place your answer for Question #7 in this cell

## Task #8: Tune n_estimators for Random Forest

Before comparing different `max_features` options, we need to choose a reasonable number of trees. Using `max_features=1/3` (one-third of predictors at each split) and `min_samples_leaf=5`, try the following values of `n_estimators`:

```python
n_trees_list = [10, 25, 50, 100, 200, 500]
```

For each value, fit a `RandomForestRegressor` with `random_state=272` and record the **test RMSE**. Plot test RMSE vs. `n_estimators`.

Based on your plot, choose a final value of `n_estimators` where the test RMSE has levelled off. State your choice.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #8 in this cell

from sklearn.ensemble import RandomForestRegressor

n_trees_list = [10, 25, 50, 100, 200, 500]
test_rmse_list = []

for n_trees in n_trees_list:
    rf = RandomForestRegressor(
        n_estimators=...,
        max_features=...,
        min_samples_leaf=5,
        random_state=272
    )
    rf.fit(..., ...)
    test_rmse_list.append(np.sqrt(mean_squared_error(y_test, rf.predict(X_test))))

plt.figure(figsize=(8, 4), dpi=150)
plt.plot(..., ..., 'o-', markersize=5)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test RMSE (log-wage scale)')
plt.title('Random Forest: Test RMSE vs. Number of Trees')
plt.tight_layout()
plt.show()

# State your chosen n_estimators
chosen_n_estimators = ...  # fill in based on plot
print(f'Chosen n_estimators: {chosen_n_estimators}')

In [ ]:
# Answer cell

from sklearn.ensemble import RandomForestRegressor

n_trees_list = [10, 25, 50, 100, 200, 500]
test_rmse_list = []

for n_trees in n_trees_list:
    rf = RandomForestRegressor(
        n_estimators=n_trees,
        max_features=1/3,
        min_samples_leaf=5,
        random_state=272
    )
    rf.fit(X_train, y_train)
    test_rmse_list.append(np.sqrt(mean_squared_error(y_test, rf.predict(X_test))))

plt.figure(figsize=(8, 4), dpi=150)
plt.plot(n_trees_list, test_rmse_list, 'o-', markersize=5)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test RMSE (log-wage scale)')
plt.title('Random Forest: Test RMSE vs. Number of Trees')
plt.tight_layout()
plt.show()

# RMSE typically stabilizes around 100-200 trees
chosen_n_estimators = 200
print(f'Chosen n_estimators: {chosen_n_estimators}')
print(f'Test RMSE values: {[f"{v:.4f}" for v in test_rmse_list]}')

## Task #9: Compare max_features Methods

A key hyperparameter in random forests is `max_features`, which controls how many predictor variables are considered at each split. Considering fewer features at each split increases the diversity among trees.

Using the `chosen_n_estimators` from Task #8 and `min_samples_leaf=5`, fit four random forests — one for each of the following `max_features` settings:

| Setting | Value | Description |
|---------|-------|-------------|
| `'sqrt'` | $\lfloor\sqrt{p}\rfloor$ | Square root of number of features |
| `'log2'` | $\lfloor\log_2 p\rfloor$ | Log base-2 of number of features |
| `1/3` | $\lfloor p/3 \rfloor$ | One-third of features (common default for regression) |
| `0.5` | $\lfloor 0.5p \rfloor$ | Half of features |

For each, compute the **test RMSE** and **test R²**. Print a summary table.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #9 in this cell

max_features_options = ['sqrt', 'log2', 1/3, 0.5]
max_features_labels  = ['sqrt', 'log2', '1/3', '0.75']

results = []
for mf, label in zip(max_features_options, max_features_labels):
    rf = RandomForestRegressor(
        n_estimators=...,
        max_features=...,
        min_samples_leaf=5,
        random_state=272
    )
    rf.fit(..., ...)
    rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
    r2   = r2_score(y_test, rf.predict(X_test))
    results.append({'max_features': label, 'Test RMSE': rmse, 'Test R²': r2})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Answer cell

max_features_options = ['sqrt', 'log2', 1/3, 0.5]
max_features_labels  = ['sqrt', 'log2', '1/3', '0.5']

results = []
rf_models = {}
for mf, label in zip(max_features_options, max_features_labels):
    rf = RandomForestRegressor(
        n_estimators=chosen_n_estimators,
        max_features=mf,
        min_samples_leaf=5,
        random_state=272
    )
    rf.fit(X_train, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
    r2   = r2_score(y_test, rf.predict(X_test))
    results.append({'max_features': label, 'Test RMSE': round(rmse, 4), 'Test R²': round(r2, 4)})
    rf_models[label] = rf

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## Task #10: Variable Importance

Using the best `max_features` setting from Task #9 (the one with the lowest test RMSE), fit a final random forest called `rf_final` and compute variable importance.

Create a horizontal bar plot of feature importances, sorted from most to least important.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #10 in this cell

# Fit the final random forest with the best max_features
best_max_features = ...  # fill in based on Task #9 results

rf_final = RandomForestRegressor(
    n_estimators=chosen_n_estimators,
    max_features=...,
    min_samples_leaf=5,
    random_state=272
)
rf_final.fit(..., ...)

# Compute variable importance
imp_df = (pd.DataFrame({'Feature': reg_feature_cols,
                         'Importance': rf_final.feature_importances_})
            .sort_values('Importance', ascending=True))

plt.figure(figsize=(8, 6), dpi=150)
plt.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue')
plt.xlabel('Gain (Mean Decrease in MSE)')
plt.title('Variable Importance — Random Forest (log INCWAGE)')
plt.tight_layout()
plt.show()

In [ ]:
# Answer cell
# (Fit using the best max_features from Task #9; here we use 1/3 as the typical default
#  for regression forests — students should pick whichever had the lowest RMSE.)

best_max_features = 1/3  # adjust based on actual results from Task #9

rf_final = RandomForestRegressor(
    n_estimators=chosen_n_estimators,
    max_features=best_max_features,
    min_samples_leaf=5,
    random_state=272
)
rf_final.fit(X_train, y_train)

imp_df = (pd.DataFrame({'Feature': reg_feature_cols,
                         'Importance': rf_final.feature_importances_})
            .sort_values('Importance', ascending=True))

plt.figure(figsize=(8, 6), dpi=150)
plt.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue')
plt.xlabel('Gain (Mean Decrease in MSE)')
plt.title('Variable Importance — Random Forest (log INCWAGE)')
plt.tight_layout()
plt.show()

print(imp_df.sort_values('Importance', ascending=False).to_string(index=False))

## Task #11: Model Comparison

Fill in the table below by running the code to print a summary comparing the three models:
- Unpruned regression tree
- Pruned regression tree (best `ccp_alpha`)
- Random forest (best `n_estimators` and `max_features`)

In [ ]:
# Place your answer for Task #11 in this cell

rmse_rf_final = np.sqrt(mean_squared_error(y_test, rf_final.predict(X_test)))
r2_rf_final   = r2_score(y_test, rf_final.predict(X_test))

comparison = pd.DataFrame({
    'Model': ['Unpruned Tree', 'Pruned Tree', 'Random Forest'],
    'Test RMSE': [rmse_test_full, rmse_pruned, rmse_rf_final],
    'Test R²':   [r2_test_full,   r2_pruned,   r2_rf_final]
})

print(comparison.to_string(index=False))

In [ ]:
# Answer cell

rmse_rf_final = np.sqrt(mean_squared_error(y_test, rf_final.predict(X_test)))
r2_rf_final   = r2_score(y_test, rf_final.predict(X_test))

comparison = pd.DataFrame({
    'Model': ['Unpruned Tree', 'Pruned Tree', 'Random Forest'],
    'Test RMSE': [round(rmse_test_full, 4), round(rmse_pruned, 4), round(rmse_rf_final, 4)],
    'Test R²':   [round(r2_test_full, 4),   round(r2_pruned, 4),   round(r2_rf_final, 4)]
})

print(comparison.to_string(index=False))

---

## Question #12: Interpreting the Results

a) Based on Task #8, how many trees are needed before the test RMSE stabilises? Explain why adding more trees beyond that point has diminishing returns.

b) Based on Task #9, does the choice of `max_features` have a large effect on test RMSE in this case? Explain conceptually why different values of `max_features` can lead to different performance.

c) Based on the variable importance plot in Task #10, which features are the most important predictors of log wages? Does this match your intuition? Briefly explain.

d) Based on the comparison table in Task #11, which model performs best on the test set? What is the cost of that improvement (in terms of model complexity and interpretability) compared to the pruned tree?

### Model Answer:

*a) Test RMSE stabilizes by 200 trees. Beyond that, adding more trees provides negligible improvement because the random variation from individual trees averages out once there are enough trees in the ensemble. Each new tree contributes less additional information since the ensemble's prediction has already converged to a stable estimate. This is the key property of averaging: variance decreases as 1/B where B is the number of trees, so gains shrink rapidly.*

**Grading note:** Students should read the stabilization point from their own plot (accept 100–200 trees as reasonable). The explanation of diminishing returns should involve the idea that more trees reduce variance but at a decreasing rate.

*b) The choice of `max_features` has a moderate effect on test RMSE in this example. Conceptually, fewer features per split (e.g., `log2`) makes individual trees more different from each other (high tree diversity) but each tree may also be weaker on its own (higher bias per tree). More features per split (e.g., `0.5`) produces stronger individual trees but with higher correlation among them, reducing the benefit of averaging. The optimal `max_features` balances tree strength and diversity. For regression problems, `1/3` is a common recommendation because it typically strikes a good balance.*

**Grading note:** Students should report whether the effect was large or small from their table, and explain the bias–variance trade-off at the tree level. Key concepts: tree diversity (decorrelation), individual tree strength, bias–variance trade-off.

*c) The most important variables are `OCC` (occupation), `AGE`, and `faminc_mid` (family income). This matches economic intuition: your occupation is the primary determinant of wage levels. Demographic variables like `AGE`, `faminc_med` are still contributing. This is different than the single pruned tree where age was at the top, indicating the influence of fitting multiple trees through RF*

**Grading note:** Students should name the top variables. The economic explanation should connect the variable to wages (e.g., occupation determines pay scale, age, family income). Differentiate RF from single tree and why this means different variable importance.

*d) The random forest performs best (lowest test RMSE, highest R²). The cost is that the random forest is a black-box model — we cannot easily visualize or interpret the decision rules the way we can with a single decision tree. The pruned tree is much more interpretable (you can trace a prediction from root to leaf) but sacrifices predictive accuracy. Whether this trade-off is worthwhile depends on the application: if the goal is prediction (e.g., estimating wages for policy analysis), the random forest is preferred; if interpretability is important (e.g., explaining individual predictions to a worker), the pruned tree may be more useful.*

**Grading note:** Students should correctly identify random forest as best-performing. The discussion of the interpretability–accuracy trade-off should be present. Accept any reasonable justification for when one might prefer the pruned tree.

Place your answer for Question #12 in this cell